In [1]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma pillow open_clip_torch torch matplotlib unstructured pydantic
import os
from textbook_loading import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
    delete_irrelevant_images,
)

In [2]:
pdf_file = './data/shortExample_PARASITES.pdf'
image_output_dir = './figures/shortExample_PARASITES'
chroma_persist_dir = './chroma/shortExample_PARASITES/'

# Make sure the data directory exists
assert os.path.exists('./data'), "Error: './data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [3]:
print("📝 Unstructuring textbooks, filtering junks, semanic chunking...")
raw_pdf_elements = load_book(pdf_file, image_output_dir)
print("🎉 1.process_pdf_with_semantic_chunking complete.")


📝 Unstructuring textbooks, filtering junks, semanic chunking...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


🎉 1.process_pdf_with_semantic_chunking complete.


In [4]:
# Clean and categorize
texts, tables, images_raw, headers_raw, titles_raw, footers_raw, figure_captions_raw, list_items_raw = clean_and_categorize_elements(raw_pdf_elements, window_size=2, min_meaningful_text_length=75)

In [5]:
# Summarize, store, etc.
text_summaries, table_summaries, image_paths, relevant_images_to_summarize, image_summaries = summarize_elements(
    texts, tables, images_raw
)

Texts and Tables Summary Done!
Checking image relevance with local textual context...
Skipping decorative image: ./figures/shortExample_PARASITES/figure-1-1.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-2-3.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-3-4.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-4-5.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-6-7.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-7-8.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-8-10.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-9-13.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-10-14.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-11-15.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-12-16.jpg
Skipping decorative image: ./figures/shortExample_PARASITES/figure-13-17.jpg
Skip

In [6]:
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables, image_paths,
    relevant_images_to_summarize, image_summaries,
    persist_directory=chroma_persist_dir
)

In [7]:
delete_irrelevant_images(images_raw, relevant_images_to_summarize)

Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-1-1.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-2-3.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-3-4.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-4-5.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-6-7.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-7-8.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-8-10.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-9-13.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-10-14.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-11-15.jpg
Successfully deleted irrelevant image: ./figures/shortExample_PARASITES/figure-12-16.jpg
Successfully deleted irrelevant ima

In [8]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

0

# Inspecting Retrieved Docs

In [9]:
query = "What are some of the most common parasites in cats?"
results = retriever.retrieve_multi_modal(query, k=5)

In [10]:
from IPython.display import display, HTML
import os

# 1. Display all images together as thumbnails
image_paths = set()
for res in results:
    if res["modality"] == "image" and os.path.exists(res["summary"]):
        image_paths.add(res["summary"])
    elif res["modality"] == "image_summary":
        img_path = res["original_metadata"].get("image_path")
        if img_path and os.path.exists(img_path):
            image_paths.add(img_path)

if image_paths:
    html_imgs = " ".join(
        f'<img src="{img}" width="100" style="margin:2px; border:1px solid #ccc;">' for img in image_paths
    )
    display(HTML(html_imgs))
else:
    print("No images found in results.")

# 2. Display original text for each text result
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "docstore"):
            doc = retriever.docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)

---------------------------------------- Retrieved Text Chunks (first 300 chars) ----------------------------------------
Ascarids (Roundworms) The cat passes eggs in her stool or larvae in her milk (1). The larvae infect her nursing kitten. Eggs from the stool (2) develop into larvae (3) and are eaten by rodents (4). The cat then eats the rodents while hunting. If the larvae pass through the kitten before maturing, th...
--------------------
Tapeworms Tapeworms are the most common internal parasite in adult cats. They live in the small intestines, and vary in length from less than 1 inch (25 mm) to sev- eral feet (1 foot is .3 meters). The scolex (head) of the parasite fastens itself to
--------------------
Ascarids (Roundworms) Ascarids are the most common worm parasite in cats, occurring in a large per- centage of kittens and in 25 to 75 percent of adults. There are two common species that infest the cat. Adult ascarids live in the stomach and intestines and can grow to 5 inches (13